SAMJAM Gemini-only experiments
=

# Setup

## Data

Input images are expected in `experiment_inputs/<folder>/<img>.jpg`.

## Notebook Setup

Google Colab
- Google Colab secrets: add a Gemini API key as `GOOGLE_API_KEY`. May be pre-populated from a previous session.
    - Need to create an account for Gemini and grab a key there.

In [1]:
import os
CONFIG_DIR = False
if not CONFIG_DIR:
    os.chdir("..")
    CONFIG_DIR = True

In [2]:
from google import genai
from PIL import Image
import json
import sys
from dotenv import load_dotenv

from common.draw import plot_bounding_boxes, draw_relationships

GOOGLE_COLAB = 'google.colab' in sys.modules

In [3]:
GEMINI_API_KEY = None
if GOOGLE_COLAB:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
else:
    load_dotenv()
    GEMINI_API_KEY = os.environ.get('GOOGLE_API_KEY')

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

## Util

In [4]:
# load dictionary of classes
with open('common/classes/classes.json') as f:
    id_bank = json.load(f)

In [5]:
def response_to_json(res):
    """
    Takes the output of a model and returns the parsed result.

    May need to clean up the output if it is a code chunk.
    """
    if res.startswith("```json"):
        res = res[7:-4]
    
    try:
        return json.loads(res)
    except:
        raise ValueError(f"Could not parse JSON: {res}")

In [6]:
def get_resized(image, max_size=1200):
    """
    Displays a resized version of the image for faster loading.

    Args:
    - image (PIL.Image): The original image.
    - max_size (int): The maximum width or height of the resized image.

    Returns:
    - PIL.Image: The resized image (does not modify the original).
    """
    # Get original size
    width, height = image.size

    # Compute new size while maintaining aspect ratio
    scale = min(max_size / width, max_size / height)
    new_size = (int(width * scale), int(height * scale))

    # Resize and show
    resized_image = image.resize(new_size, Image.Resampling.LANCZOS)
    # resized_image.show()

    return resized_image

In [7]:
with open("prompts/curframe.txt", "r") as f:
    curframe_prompt = f.read()
with open("prompts/curframe_noids.txt", "r") as f:
    curframe_noids_prompt = f.read()
with open("prompts/new_curframe_only.txt", "r") as f:
    new_curframe_only_prompt = f.read()

def generate_scene_graph(img, model="gemini-1.5-flash", prev_img=None, prev_sg=[], with_ids=True):
    contents = [
        img,
        # curframe_prompt if with_ids else curframe_noids_prompt,
        new_curframe_only_prompt
    ]

    if with_ids:
        contents.append(json.dumps(id_bank))
    
    response = gemini_client.models.generate_content(
        model=model,
        contents=contents,
        config={
            "response_mime_type": "application/json"
        }
    ).text

    graph = response_to_json(response)
    return graph

In [8]:
with open("prompts/curframe_prevsg.txt", "r") as f:
    curframe_prevsg_prompt = f.read()
with open("prompts/curframe_prevsg_noids.txt", "r") as f:
    curframe_prevsg_noids_prompt = f.read()
with open("prompts/new_curframe_prevsg.txt", "r") as f:
    new_curframe_prevsg_prompt = f.read()

def generate_scene_graph_with_prevsg(img, model="gemini-1.5-flash", prev_img=None, prev_sg=[], with_ids=True):
    contents = [
        img,
        # curframe_prevsg_prompt if with_ids else curframe_prevsg_noids_prompt,
        new_curframe_prevsg_prompt
    ]

    if with_ids:
        contents.append(json.dumps(id_bank))

    if prev_sg:
        for psg in prev_sg:
            contents.append(json.dumps(psg))

    response = gemini_client.models.generate_content(
        model=model,
        contents=contents,
        config={
            "response_mime_type": "application/json"
        }
    ).text

    graph = response_to_json(response)
    return graph

In [9]:
with open("prompts/new_curframe_allprevsg.txt", "r") as f:
    new_curframe_allprevsg_prompt = f.read()

def generate_scene_graph_with_allprevsg(img, model="gemini-1.5-flash", prev_img=None, prev_sg=[], with_ids=True):
    contents = [
        img,
        new_curframe_allprevsg_prompt
    ]

    if prev_sg:
        for psg in prev_sg:
            contents.append(json.dumps(psg))

    response = gemini_client.models.generate_content(
        model=model,
        contents=contents,
        config={
            "response_mime_type": "application/json"
        }
    ).text

    graph = response_to_json(response)
    return graph

## Load Data

In [10]:
experiment_inputs = {}

path = "samjam/experiment_inputs"
experiment_folders = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]

# there are ~11 images in each experiment folder. load them into the dict with the name of the file
# the name of the file is not an integer, you should find the filename.
for folder in experiment_folders:
    experiment_inputs[folder] = []
    for file in os.listdir(f"{path}/{folder}"):
        if file.endswith(".jpg"):
            experiment_inputs[folder].append(file)

print("Experiments:")
for folder in experiment_folders:
    print(f"- {folder} ({len(experiment_inputs[folder])} images)")

Experiments:
- cut_bell_pepper (11 images)
- pick_up_cereal (11 images)
- put_cup (5 images)
- throw_carton (11 images)


In [ ]:
# helper for going through the experiments
def run_experiment(
    generate_fn,
    experiment_name="curframe_only",
    model="gemini-1.5-flash",
    output_folder="samjam/output/",
    force_rerun=False,
    with_ids=True,
    first_generate_fn=None,
    transform=None,
):
    """
    Runs the scene function on each scene folder.

    :param f: The function to run on each image. It takes the current image, prev image, prev scene graph.
    """

    print(f"Using model {model}")

    for scene in experiment_folders:
        print(f"Running scene {scene}...")
        experiment_output = f"{output_folder}/{model}/{experiment_name}/{scene}"
        print(f"Output folder: {experiment_output}")
        total_img = len(experiment_inputs[scene])
        print(f"Total images: {total_img}")
        print("." * total_img)

        # create the output folder
        os.makedirs(f"{experiment_output}/", exist_ok=True)

        is_first_frame = True
        prev_sg = []
        prev_img = None

        for file in sorted(experiment_inputs[scene]):
            # if output exists, skip
            if not force_rerun and os.path.exists(f"{experiment_output}/{file}"):
                print("-", end="")
                # get prev sg and prev img
                with open(f"{experiment_output}/{file}.json", "r") as f:
                    prev_sg.append(json.load(f))
                with Image.open(f"{experiment_output}/{file}") as image:
                    prev_img = get_resized(image)
                continue

            # generate output
            print(".", end="")
            with Image.open(f"{path}/{scene}/{file}") as image:
                resized_image = get_resized(image)

                sg = None
                if is_first_frame and first_generate_fn:
                    sg = first_generate_fn(resized_image, model, with_ids)
                    is_first_frame = False
                else:
                    # print(f"feeding {len(prev_sg)} graphs")
                    sg = generate_fn(resized_image, model, prev_img, prev_sg, with_ids)

                if transform:
                    old_sg = sg
                    sg = transform(sg)

                prev_sg.append(sg)
                prev_img = resized_image

                # save the scene graph as json
                with open(f"{experiment_output}/{file}.json", "w") as f:
                    json.dump(old_sg, f)
                with open(f"{experiment_output}/{file}_indented.json", "w") as f:
                    json.dump(old_sg, f, indent=2)

                # draw the scene graphs and save the images
                plot_bounding_boxes(resized_image, sg)
                draw_relationships(resized_image, sg, with_ids=with_ids)
                resized_image.save(f"{experiment_output}/{file}")

            is_first_frame = False

        print()

# Current frame only

In [11]:
# run_experiment(generate_scene_graph, "curframe_only", model="gemini-1.5-flash")

In [12]:
# run_experiment(generate_scene_graph, "curframe_only", model="gemini-2.0-flash")

In [13]:
# run_experiment(generate_scene_graph_with_prevsg, "curframe_prevsg", model="gemini-2.0-flash")

In [14]:
# run_experiment(generate_scene_graph, "curframe_only_noids", model="gemini-2.0-flash", with_ids=False)
# run_experiment(generate_scene_graph_with_prevsg, "curframe_prevsg_noids", model="gemini-2.0-flash", with_ids=False)

## New prompt

In [12]:
# map from new format to old format
def transform(new_graph):
    old_graph = {
        "objects": {},
        "relations": []
    }

    for obj in new_graph["objects"]:
        old_graph["objects"][obj["id"]] = {
            "category": obj["name"],
            "bounding_box": obj["bbox"]
        }
    
    for relationship in new_graph["relationships"]:
        rel = (relationship["subj_id"], relationship["obj_id"], relationship["predicate"])
        old_graph["relations"].append(rel)
    
    return old_graph

In [13]:
run_experiment(generate_scene_graph, "new_curframe_only_noids", model="gemini-2.0-flash", with_ids=False, transform=transform)
run_experiment(generate_scene_graph_with_prevsg, "new_curframe_prevsg_noids", model="gemini-2.0-flash", with_ids=False, first_generate_fn=generate_scene_graph, transform=transform)

Using model gemini-2.0-flash
Running scene cut_bell_pepper...
Output folder: samjam/output//gemini-2.0-flash/new_curframe_only_noids/cut_bell_pepper
Total images: 11
...........
-----------
Running scene pick_up_cereal...
Output folder: samjam/output//gemini-2.0-flash/new_curframe_only_noids/pick_up_cereal
Total images: 11
...........
-----------
Running scene put_cup...
Output folder: samjam/output//gemini-2.0-flash/new_curframe_only_noids/put_cup
Total images: 5
.....
-----
Running scene throw_carton...
Output folder: samjam/output//gemini-2.0-flash/new_curframe_only_noids/throw_carton
Total images: 11
...........
-----------
Using model gemini-2.0-flash
Running scene cut_bell_pepper...
Output folder: samjam/output//gemini-2.0-flash/new_curframe_prevsg_noids/cut_bell_pepper
Total images: 11
...........
-----------
Running scene pick_up_cereal...
Output folder: samjam/output//gemini-2.0-flash/new_curframe_prevsg_noids/pick_up_cereal
Total images: 11
...........
-----------
Running sce

# Test all previous scene graphs

In [19]:
run_experiment(generate_scene_graph_with_allprevsg, "new_curframe_allprevsg", model="gemini-2.0-flash", with_ids=False, transform=transform)

Using model gemini-2.0-flash
Running scene cut_bell_pepper...
Output folder: samjam/output//gemini-2.0-flash/new_curframe_allprevsg/cut_bell_pepper
Total images: 11
...........
-----------
Running scene pick_up_cereal...
Output folder: samjam/output//gemini-2.0-flash/new_curframe_allprevsg/pick_up_cereal
Total images: 11
...........
-----------
Running scene put_cup...
Output folder: samjam/output//gemini-2.0-flash/new_curframe_allprevsg/put_cup
Total images: 5
.....
----.feeding 4 graphs

Running scene throw_carton...
Output folder: samjam/output//gemini-2.0-flash/new_curframe_allprevsg/throw_carton
Total images: 11
...........
-----------
